# Replication: Begenau, Liang & Siriwardane (2026) — Figures & Tables

This notebook replicates the main-paper figures and tables from *"The Rise of Alternatives"* (RFS 2026) that are feasible with the available PPD data.

**Already replicated elsewhere:** Table 1, Figure 2 (see `Begenau replication work/replicate_table1_figure2.ipynb`)

**Replicated here:**
- Figure 1a — Aggregate Risky Share
- Figure 3a — Cross-Section of Omega (Box Plots)
- Figure 3b — Distribution of Delta Omega (Histogram)
- Figure 3c — Within-Alternatives Heterogeneity
- Table 7 — Risk-Seeking Motives (OLS)
- Figure 6a — Constraint Tightness vs Delta Omega
- Figure 6b — Cash Share vs Delta Omega
- Figure A5a/b — Risky Share Distribution (Internet Appendix)
- Table A1a/b — Transition Matrices (Internet Appendix)

## Section 0: Setup & Variable Construction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import statsmodels.api as sm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Plot style — match existing figure_2.png
plt.rcParams.update({
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

COLORS = {
    'blue': '#1f77b4',
    'orange': '#d35400',
    'purple': '#7f6d9e',
    'light_blue': '#aec7e8',
    'red': '#d62728',
}

OUTPUT_DIR = Path('../output_python/replication_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load cleaned plan-level data
df = pd.read_csv('../output_python/ppd_plan_level_clean.csv')
print(f'Plan-level data: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Fiscal years: {df["fy"].min():.0f} - {df["fy"].max():.0f}')
print(f'Unique plans (ppd_id): {df["ppd_id"].nunique()}')
print(f'Unique systems (pub_id): {df["pub_id"].nunique()}')

In [ ]:
# --- Construct derived variables on plan-level data ---

# Asset class aggregation — TARGET shares
df['real_assets_trgt'] = df['RETotal_Trgt'].fillna(0) + df['COMDTotal_Trgt'].fillna(0)
df['alternatives_trgt'] = (
    df['PETotal_Trgt'].fillna(0)
    + df['real_assets_trgt']
    + df['HFTotal_Trgt'].fillna(0)
    + df['AltMiscTotal_Trgt'].fillna(0)
    + df['OtherTotal_Trgt'].fillna(0)
)
df['public_equity_trgt'] = df['EQTotal_Trgt'].fillna(0)
df['fixed_income_trgt'] = df['FITotal_Trgt'].fillna(0) + df['CashTotal_Trgt'].fillna(0)
df['risky_trgt'] = df['public_equity_trgt'] + df['alternatives_trgt']

# Asset class aggregation — ACTUAL shares
df['real_assets_actl'] = df['RETotal_Actl'].fillna(0) + df['COMDTotal_Actl'].fillna(0)
df['alternatives_actl'] = (
    df['PETotal_Actl'].fillna(0)
    + df['real_assets_actl']
    + df['HFTotal_Actl'].fillna(0)
    + df['AltMiscTotal_Actl'].fillna(0)
    + df['OtherTotal_Actl'].fillna(0)
)
df['public_equity_actl'] = df['EQTotal_Actl'].fillna(0)
df['risky_actl'] = df['public_equity_actl'] + df['alternatives_actl']

# Omega (alternative-to-risky share, target)
denom = df['alternatives_trgt'] + df['public_equity_trgt']
df['omega'] = np.where(denom > 0, df['alternatives_trgt'] / denom, np.nan)

# Constraint slack
df['l_pt'] = df['risky_actl'] - df['risky_trgt']

# Pct retired
df['pct_retired'] = np.where(
    df['TotMembership'] > 0,
    df['beneficiaries_tot'] / df['TotMembership'],
    np.nan
)

# Rename for convenience
df['hurdle_rate'] = df['InvestmentReturnAssumption_GASB']
df['gasb25_funding'] = df['ActFundedRatio_GASB']
df['cash_share_trgt'] = df['CashTotal_Trgt'].fillna(0)

print('Derived variables constructed.')

In [ ]:
# --- System-level aggregation ---
# The paper works at the pension system level (pub_id), not individual plans.
# Aggregate using AUM-weighted averages for portfolio shares and ratios,
# sums for membership/AUM.

# Columns to AUM-weight-average
share_cols = [
    'EQTotal_Trgt', 'FITotal_Trgt', 'RETotal_Trgt', 'PETotal_Trgt',
    'HFTotal_Trgt', 'AltMiscTotal_Trgt', 'COMDTotal_Trgt', 'CashTotal_Trgt',
    'OtherTotal_Trgt',
    'EQTotal_Actl', 'FITotal_Actl', 'RETotal_Actl', 'PETotal_Actl',
    'HFTotal_Actl', 'AltMiscTotal_Actl', 'COMDTotal_Actl', 'CashTotal_Actl',
    'OtherTotal_Actl',
    'alternatives_trgt', 'public_equity_trgt', 'fixed_income_trgt',
    'risky_trgt', 'real_assets_trgt',
    'alternatives_actl', 'public_equity_actl', 'risky_actl', 'real_assets_actl',
    'omega', 'l_pt', 'cash_share_trgt',
    'gasb25_funding', 'hurdle_rate', 'pct_retired',
    'InvestmentReturn_1yr',
]

# Columns to sum
sum_cols = ['MktAssets_net', 'TotMembership', 'beneficiaries_tot', 'actives_tot']

# Columns to take first (identifiers)
first_cols = ['pub_system_name', 'StateAbbrev', 'region', 'division',
              'general_consultant', 'consultant_id', 'ppd_zipcode']

def weighted_avg(group, col, weight_col='MktAssets_net'):
    """Compute AUM-weighted average, handling NaNs."""
    mask = group[col].notna() & group[weight_col].notna() & (group[weight_col] > 0)
    if mask.sum() == 0:
        return np.nan
    return np.average(group.loc[mask, col], weights=group.loc[mask, weight_col])

rows = []
for (pub_id, fy), grp in df.groupby(['pub_id', 'fy']):
    row = {'pub_id': pub_id, 'fy': fy}
    # Weighted averages
    for col in share_cols:
        if col in grp.columns:
            row[col] = weighted_avg(grp, col)
    # Sums
    for col in sum_cols:
        if col in grp.columns:
            row[col] = grp[col].sum()
    # First values
    for col in first_cols:
        if col in grp.columns:
            row[col] = grp[col].iloc[0]
    # ret_bgnassets — take from performance file (already at system level)
    if 'ret_bgnassets' in grp.columns:
        row['ret_bgnassets'] = weighted_avg(grp, 'ret_bgnassets')
    rows.append(row)

sys_df = pd.DataFrame(rows)

# Recompute pct_retired at system level from summed membership
sys_df['pct_retired'] = np.where(
    sys_df['TotMembership'] > 0,
    sys_df['beneficiaries_tot'] / sys_df['TotMembership'],
    np.nan
)

print(f'System-level data: {sys_df.shape[0]} system-year observations')
print(f'Unique systems: {sys_df["pub_id"].nunique()}')
print(f'Fiscal years: {sys_df["fy"].min():.0f} - {sys_df["fy"].max():.0f}')

In [ ]:
# --- Long-difference variables (for Table 7 and Figures 6a/b) ---
# Systems present in both 2002 and 2021

vars_for_diff = ['omega', 'gasb25_funding', 'hurdle_rate', 'pct_retired',
                 'cash_share_trgt', 'risky_trgt']

df_2002 = sys_df[sys_df['fy'] == 2002].set_index('pub_id')[vars_for_diff + ['StateAbbrev']]
df_2021 = sys_df[sys_df['fy'] == 2021].set_index('pub_id')[vars_for_diff]

# Long differences
delta = (df_2021 - df_2002).rename(columns=lambda c: f'delta_{c}')
delta['StateAbbrev'] = df_2002['StateAbbrev']

# l_bar: time-average of constraint slack for each system
l_bar = sys_df.groupby('pub_id')['l_pt'].mean().rename('l_bar')
delta = delta.join(l_bar)

# Drop systems with missing omega change
delta = delta.dropna(subset=['delta_omega'])

print(f'Long-difference sample: {len(delta)} systems (present in both 2002 and 2021)')
print(f'States represented: {delta["StateAbbrev"].nunique()}')

In [ ]:
# --- Verification: observation count ---
n_sys_yr = sys_df.shape[0]
print(f'System-year observations: {n_sys_yr}')
print(f'Paper target: 3,128')
print(f'Match: {"YES" if n_sys_yr == 3128 else "NO — difference of " + str(n_sys_yr - 3128)}')

# Quick sanity: aggregate omega endpoints
for yr in [2001, 2021]:
    sub = sys_df[sys_df['fy'] == yr]
    w = sub['MktAssets_net']
    mask = sub['omega'].notna() & w.notna() & (w > 0)
    omega_agg = np.average(sub.loc[mask, 'omega'], weights=w[mask])
    risky_agg = np.average(sub.loc[mask, 'risky_trgt'], weights=w[mask])
    print(f'{yr}: omega={omega_agg:.3f} (paper: {0.135 if yr==2001 else 0.395}), '
          f'risky={risky_agg:.3f} (paper: {0.69 if yr==2001 else 0.76})')

## Section 1: Figure 1a — Aggregate Risky Share

In [ ]:
# Compute asset-weighted aggregate risky share by year
risky_agg = []
for yr in range(2001, 2022):
    sub = sys_df[sys_df['fy'] == yr]
    mask = sub['risky_trgt'].notna() & sub['MktAssets_net'].notna() & (sub['MktAssets_net'] > 0)
    if mask.sum() > 0:
        val = np.average(sub.loc[mask, 'risky_trgt'], weights=sub.loc[mask, 'MktAssets_net'])
        risky_agg.append({'fy': yr, 'risky_share': val})

risky_agg = pd.DataFrame(risky_agg)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(risky_agg['fy'], risky_agg['risky_share'] * 100,
        marker='o', lw=2, color=COLORS['blue'], markersize=5)
ax.set_xlabel('Year')
ax.set_ylabel('Target Risky Share (%)')
ax.set_title('Figure 1a \u2014 Aggregate Risky Share')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.set_xlim(2000.5, 2021.5)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_1a.png')
risky_agg.to_csv(OUTPUT_DIR / 'figure_1a_data.csv', index=False)
plt.show()

print(f'2001 risky share: {risky_agg.iloc[0]["risky_share"]:.3f} (paper: 0.69)')
print(f'2021 risky share: {risky_agg.iloc[-1]["risky_share"]:.3f} (paper: 0.76)')

## Section 2: Figure 3a — Cross-Section of Omega Over Time

In [ ]:
# Box plots of omega for even years 2002-2020 + 2021
# Whiskers at p10/p90 (not default IQR)

years_3a = list(range(2002, 2021, 2)) + [2021]

# Compute custom stats for each year
box_stats = []
for yr in years_3a:
    vals = sys_df.loc[sys_df['fy'] == yr, 'omega'].dropna() * 100
    stats = {
        'med': vals.median(),
        'q1': vals.quantile(0.25),
        'q3': vals.quantile(0.75),
        'whislo': vals.quantile(0.10),
        'whishi': vals.quantile(0.90),
        'fliers': [],
        'label': str(yr),
    }
    box_stats.append(stats)

fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.bxp(box_stats, showfliers=False, patch_artist=True,
            medianprops=dict(color='black', lw=1.5),
            boxprops=dict(facecolor=COLORS['light_blue'], edgecolor=COLORS['blue']),
            whiskerprops=dict(color=COLORS['blue'], linestyle='--'),
            capprops=dict(color=COLORS['blue']))
ax.set_xlabel('Year')
ax.set_ylabel('Alternative-to-Risky Share (%)')
ax.set_title('Figure 3a \u2014 Cross-Section of Omega Over Time')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_3a.png')
plt.show()

# Verification
vals_2021 = sys_df.loc[sys_df['fy'] == 2021, 'omega'].dropna()
print(f'2021 p10={vals_2021.quantile(0.10):.2%} (paper: 18%), '
      f'p90={vals_2021.quantile(0.90):.2%} (paper: 57%)')

## Section 3: Figure 3b — Distribution of Delta Omega

In [ ]:
# Histogram of omega changes from 2002 to 2021
delta_omega_pp = delta['delta_omega'].dropna() * 100  # in percentage points

fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(delta_omega_pp, bins=25, color=COLORS['blue'], edgecolor='white', alpha=0.85)

# Annotate percentiles
p25 = delta_omega_pp.quantile(0.25)
p50 = delta_omega_pp.median()
p75 = delta_omega_pp.quantile(0.75)
for pval, label in [(p25, 'p25'), (p50, 'Median'), (p75, 'p75')]:
    ax.axvline(pval, color=COLORS['red'], linestyle='--', lw=1.5, alpha=0.7)
    ax.text(pval, ax.get_ylim()[1] * 0.92, f'{label}={pval:.0f}pp',
            ha='center', fontsize=9, color=COLORS['red'])

ax.set_xlabel('Change in Omega (pp), 2002\u20132021')
ax.set_ylabel('Number of Systems')
ax.set_title('Figure 3b \u2014 Distribution of \u0394\u03C9 (2002\u20132021)')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_3b.png')
plt.show()

print(f'p25={p25:.1f}pp (paper: +15pp), p75={p75:.1f}pp (paper: +35pp)')
print(f'N systems: {len(delta_omega_pp)}')

## Section 4: Figure 3c — Within-Alternatives Heterogeneity (2021)

In [ ]:
# Box plots of PE/alt, RA/alt, HF/alt shares in 2021
df_2021_sys = sys_df[sys_df['fy'] == 2021].copy()
df_2021_sys = df_2021_sys[df_2021_sys['alternatives_trgt'] > 0.01]  # avoid near-zero denominators

df_2021_sys['pe_to_alt'] = df_2021_sys['PETotal_Trgt'].fillna(0) / df_2021_sys['alternatives_trgt'] * 100
df_2021_sys['ra_to_alt'] = df_2021_sys['real_assets_trgt'] / df_2021_sys['alternatives_trgt'] * 100
df_2021_sys['hf_to_alt'] = df_2021_sys['HFTotal_Trgt'].fillna(0) / df_2021_sys['alternatives_trgt'] * 100

subcats = ['pe_to_alt', 'ra_to_alt', 'hf_to_alt']
labels = ['PE/Credit', 'Real Assets', 'Hedge Funds']

box_stats_3c = []
for col, label in zip(subcats, labels):
    vals = df_2021_sys[col].dropna()
    stats = {
        'med': vals.median(),
        'q1': vals.quantile(0.25),
        'q3': vals.quantile(0.75),
        'whislo': vals.quantile(0.10),
        'whishi': vals.quantile(0.90),
        'fliers': [],
        'label': label,
    }
    box_stats_3c.append(stats)

fig, ax = plt.subplots(figsize=(6, 5))
bp = ax.bxp(box_stats_3c, showfliers=False, patch_artist=True,
            medianprops=dict(color='black', lw=1.5),
            boxprops=dict(facecolor=COLORS['light_blue'], edgecolor=COLORS['blue']),
            whiskerprops=dict(color=COLORS['blue'], linestyle='--'),
            capprops=dict(color=COLORS['blue']))
ax.set_ylabel('Share of Alternatives (%)')
ax.set_title('Figure 3c \u2014 Within-Alternatives Heterogeneity (2021)')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_3c.png')
plt.show()

pe_vals = df_2021_sys['pe_to_alt'].dropna()
print(f'PE/alt p25={pe_vals.quantile(0.25):.0f}% (paper: 20%), '
      f'p75={pe_vals.quantile(0.75):.0f}% (paper: 40%)')

## Section 5: Table 7 — Risk-Seeking Motives

In [ ]:
# Table 7: Single-variable OLS regressions of delta_omega on risk-seeking proxies
# SE clustered by state

def run_ols_clustered(y, X, cluster_var):
    """Run OLS with state-clustered standard errors."""
    mask = y.notna()
    for col in X.columns:
        mask &= X[col].notna()
    mask &= cluster_var.notna()
    y_clean = y[mask]
    X_clean = sm.add_constant(X[mask])
    cluster_clean = cluster_var[mask]
    model = sm.OLS(y_clean, X_clean).fit(
        cov_type='cluster',
        cov_kwds={'groups': cluster_clean}
    )
    return model

# Run 6 regressions (skip col 2 which needs BEA data)
results = {}

y = delta['delta_omega']
cluster = delta['StateAbbrev']

# Col 1: delta_gasb25_funding (no controls)
results['(1)'] = run_ols_clustered(y, delta[['delta_gasb25_funding']], cluster)

# Col 3: delta_hurdle_rate (no controls)
results['(3)'] = run_ols_clustered(y, delta[['delta_hurdle_rate']], cluster)

# Col 4: delta_pct_retired (no controls)
results['(4)'] = run_ols_clustered(y, delta[['delta_pct_retired']], cluster)

# Col 5: delta_gasb25_funding + controls (l_bar, delta_cash)
results['(5)'] = run_ols_clustered(
    y, delta[['delta_gasb25_funding', 'l_bar', 'delta_cash_share_trgt']], cluster)

# Col 6: delta_hurdle_rate + controls
results['(6)'] = run_ols_clustered(
    y, delta[['delta_hurdle_rate', 'l_bar', 'delta_cash_share_trgt']], cluster)

# Col 7: delta_pct_retired + controls
results['(7)'] = run_ols_clustered(
    y, delta[['delta_pct_retired', 'l_bar', 'delta_cash_share_trgt']], cluster)

print('Regressions complete.')

In [ ]:
# Format Table 7 output

# Paper reference values for comparison
paper_ref = {
    '(1)': {'var': 'delta_gasb25_funding', 'beta': -0.10, 't': -1.16, 'R2': 0.01, 'N': 118},
    '(3)': {'var': 'delta_hurdle_rate', 'beta': -1.02, 't': -0.31, 'R2': 0.00, 'N': 117},
    '(4)': {'var': 'delta_pct_retired', 'beta': 0.13, 't': 0.76, 'R2': 0.00, 'N': 115},
}

rows_table = []
for col_name, model in results.items():
    # Get the main variable (first non-constant regressor)
    main_var = [v for v in model.params.index if v != 'const'][0]
    beta = model.params[main_var]
    tstat = model.tvalues[main_var]
    pval = model.pvalues[main_var]
    stars = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.10 else ''
    rows_table.append({
        'Column': col_name,
        'Main Variable': main_var,
        'Beta': f'{beta:.2f}{stars}',
        't-stat': f'({tstat:.2f})',
        'R-squared': f'{model.rsquared:.2f}',
        'N': int(model.nobs),
        'Controls': 'Yes' if model.df_model > 1 else 'No',
    })

table7 = pd.DataFrame(rows_table).set_index('Column')
table7.to_csv(OUTPUT_DIR / 'table_7.csv')

print('Table 7 — Risk-Seeking Motives')
print('=' * 80)
print(table7.to_string())
print('\nPaper reference (no-control specs):')
for col, ref in paper_ref.items():
    print(f'  {col}: beta={ref["beta"]}, t={ref["t"]}, R2={ref["R2"]}, N={ref["N"]}')

## Section 6: Figure 6a — Constraint Tightness vs Delta Omega

In [ ]:
# Binscatter: X = l_bar (avg actual-target risky gap), Y = delta_omega
# 20 equal-frequency bins

def binscatter(x, y, n_bins=20, ax=None, xlabel='', ylabel='', title='',
               color=None, save_path=None):
    """Create a binscatter plot with OLS line."""
    data = pd.DataFrame({'x': x, 'y': y}).dropna()
    if len(data) < n_bins:
        n_bins = len(data) // 2
    data['bin'] = pd.qcut(data['x'], n_bins, labels=False, duplicates='drop')
    binned = data.groupby('bin')[['x', 'y']].mean()

    # OLS on raw data
    X_ols = sm.add_constant(data['x'])
    model = sm.OLS(data['y'], X_ols).fit(cov_type='HC1')
    beta = model.params['x']
    tstat = model.tvalues['x']
    r2 = model.rsquared
    n = int(model.nobs)

    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))
    else:
        fig = ax.get_figure()

    c = color or COLORS['blue']
    ax.scatter(binned['x'] * 100, binned['y'] * 100, s=60, color=c, zorder=5)

    # Regression line
    x_line = np.linspace(data['x'].min(), data['x'].max(), 100)
    y_line = model.params['const'] + beta * x_line
    ax.plot(x_line * 100, y_line * 100, '--', color=COLORS['red'], lw=1.5, alpha=0.8)

    # Annotation
    ax.text(0.05, 0.95, f'\u03B2 = {beta:.2f}\nt = {tstat:.2f}\nR\u00B2 = {r2:.2f}\nN = {n}',
            transform=ax.transAxes, va='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis='both', alpha=0.3)

    if save_path:
        fig.tight_layout()
        fig.savefig(save_path)

    return model

fig, ax = plt.subplots(figsize=(7, 5))
model_6a = binscatter(
    delta['l_bar'], delta['delta_omega'],
    ax=ax,
    xlabel='Avg. Actual\u2013Target Risky Gap (pp)',
    ylabel='\u0394\u03C9 (pp), 2002\u20132021',
    title='Figure 6a \u2014 Constraint Tightness vs \u0394\u03C9',
    save_path=OUTPUT_DIR / 'figure_6a.png',
)
plt.show()
print(f'Paper: beta=1.34, t=1.78, R2=0.04')

## Section 7: Figure 6b — Cash Share vs Delta Omega

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
model_6b = binscatter(
    delta['delta_cash_share_trgt'], delta['delta_omega'],
    ax=ax,
    xlabel='\u0394 Cash Share (pp)',
    ylabel='\u0394\u03C9 (pp), 2002\u20132021',
    title='Figure 6b \u2014 \u0394 Cash Share vs \u0394\u03C9',
    save_path=OUTPUT_DIR / 'figure_6b.png',
)
plt.show()
print(f'Paper: beta=-0.74, t=-1.49, R2=0.02')

## Section 8: Figure A5a/b — Risky Share Distribution (Internet Appendix)

In [ ]:
# Figure A5a: Box plots of risky_share_trgt (same structure as Figure 3a)
# Figure A5b: Histogram of delta_risky_share (same structure as Figure 3b)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- A5a: Risky share box plots ---
years_a5 = list(range(2002, 2021, 2)) + [2021]
box_stats_a5 = []
for yr in years_a5:
    vals = sys_df.loc[sys_df['fy'] == yr, 'risky_trgt'].dropna() * 100
    stats = {
        'med': vals.median(),
        'q1': vals.quantile(0.25),
        'q3': vals.quantile(0.75),
        'whislo': vals.quantile(0.10),
        'whishi': vals.quantile(0.90),
        'fliers': [],
        'label': str(yr),
    }
    box_stats_a5.append(stats)

axes[0].bxp(box_stats_a5, showfliers=False, patch_artist=True,
            medianprops=dict(color='black', lw=1.5),
            boxprops=dict(facecolor=COLORS['light_blue'], edgecolor=COLORS['blue']),
            whiskerprops=dict(color=COLORS['blue'], linestyle='--'),
            capprops=dict(color=COLORS['blue']))
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Target Risky Share (%)')
axes[0].set_title('Figure A5a \u2014 Risky Share Distribution')
axes[0].grid(axis='y', alpha=0.3)

# --- A5b: Delta risky share histogram ---
delta_risky_pp = delta['delta_risky_trgt'].dropna() * 100
axes[1].hist(delta_risky_pp, bins=25, color=COLORS['blue'], edgecolor='white', alpha=0.85)
p25_r = delta_risky_pp.quantile(0.25)
p75_r = delta_risky_pp.quantile(0.75)
for pval, label in [(p25_r, 'p25'), (p75_r, 'p75')]:
    axes[1].axvline(pval, color=COLORS['red'], linestyle='--', lw=1.5, alpha=0.7)
    axes[1].text(pval, axes[1].get_ylim()[1] * 0.92 if axes[1].get_ylim()[1] > 0 else 5,
                 f'{label}={pval:.0f}pp', ha='center', fontsize=9, color=COLORS['red'])
axes[1].set_xlabel('\u0394 Risky Share (pp), 2002\u20132021')
axes[1].set_ylabel('Number of Systems')
axes[1].set_title('Figure A5b \u2014 Distribution of \u0394 Risky Share')
axes[1].grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_a5.png')
plt.show()

## Section 9: Table A1a/b — Transition Matrices

In [ ]:
# Quartile transition matrices from 2002 to 2021

def transition_matrix(var_name, var_label):
    """Compute quartile transition matrix from 2002 to 2021."""
    d2002 = sys_df[sys_df['fy'] == 2002][['pub_id', var_name]].dropna().set_index('pub_id')
    d2021 = sys_df[sys_df['fy'] == 2021][['pub_id', var_name]].dropna().set_index('pub_id')
    both = d2002.join(d2021, lsuffix='_2002', rsuffix='_2021').dropna()

    both['Q_2002'] = pd.qcut(both[f'{var_name}_2002'], 4, labels=['Q1','Q2','Q3','Q4'])
    both['Q_2021'] = pd.qcut(both[f'{var_name}_2021'], 4, labels=['Q1','Q2','Q3','Q4'])

    trans = pd.crosstab(both['Q_2002'], both['Q_2021'], normalize='index') * 100
    trans.index.name = f'{var_label} Quartile (2002)'
    trans.columns.name = f'{var_label} Quartile (2021)'
    return trans

# Table A1a: Risky share
trans_risky = transition_matrix('risky_trgt', 'Risky Share')
print('Table A1a — Risky Share Transition Matrix (2002 → 2021)')
print(trans_risky.round(0).to_string())
print()

# Table A1b: Omega
trans_omega = transition_matrix('omega', 'Alt-to-Risky (ω)')
print('Table A1b — Alt-to-Risky Transition Matrix (2002 → 2021)')
print(trans_omega.round(0).to_string())

trans_risky.to_csv(OUTPUT_DIR / 'table_a1a_transition_risky.csv')
trans_omega.to_csv(OUTPUT_DIR / 'table_a1b_transition_omega.csv')

## Section 10: Verification Summary

In [ ]:
# Collect all verification checks
checks = []

# System-year count
checks.append(('Obs count', 'System-year observations', sys_df.shape[0], 3128))

# Figure 1a endpoints
checks.append(('Fig 1a', 'Risky share 2001',
               round(risky_agg.iloc[0]['risky_share'] * 100, 1), 69.0))
checks.append(('Fig 1a', 'Risky share 2021',
               round(risky_agg.iloc[-1]['risky_share'] * 100, 1), 76.0))

# Figure 3a whiskers
vals_2021 = sys_df.loc[sys_df['fy'] == 2021, 'omega'].dropna()
checks.append(('Fig 3a', 'Omega p10 (2021)',
               round(vals_2021.quantile(0.10) * 100, 0), 18.0))
checks.append(('Fig 3a', 'Omega p90 (2021)',
               round(vals_2021.quantile(0.90) * 100, 0), 57.0))

# Figure 3b percentiles
checks.append(('Fig 3b', 'Delta omega p25 (pp)',
               round(delta['delta_omega'].dropna().quantile(0.25) * 100, 0), 15.0))
checks.append(('Fig 3b', 'Delta omega p75 (pp)',
               round(delta['delta_omega'].dropna().quantile(0.75) * 100, 0), 35.0))

# Figure 3c PE quartiles
pe_vals = df_2021_sys['pe_to_alt'].dropna()
checks.append(('Fig 3c', 'PE/alt p25 (2021)',
               round(pe_vals.quantile(0.25), 0), 20.0))
checks.append(('Fig 3c', 'PE/alt p75 (2021)',
               round(pe_vals.quantile(0.75), 0), 40.0))

# Table 7 col 1
if '(1)' in results:
    m = results['(1)']
    main_var = [v for v in m.params.index if v != 'const'][0]
    checks.append(('Table 7', 'Col 1 beta',
                   round(m.params[main_var], 2), -0.10))
    checks.append(('Table 7', 'Col 1 N',
                   int(m.nobs), 118))

# Figure 6a
checks.append(('Fig 6a', 'Beta (l_bar)',
               round(model_6a.params['x'], 2), 1.34))

# Figure 6b
checks.append(('Fig 6b', 'Beta (delta cash)',
               round(model_6b.params['x'], 2), -0.74))

verify_df = pd.DataFrame(checks, columns=['Section', 'Metric', 'Replicated', 'Paper'])
verify_df['Gap'] = verify_df['Replicated'] - verify_df['Paper']
verify_df['Match'] = verify_df['Gap'].abs().apply(
    lambda x: 'EXACT' if x == 0 else ('CLOSE' if x <= 2 else 'CHECK'))

print('Verification Summary')
print('=' * 80)
print(verify_df.to_string(index=False))
verify_df.to_csv(OUTPUT_DIR / 'verification_summary.csv', index=False)

## Section 11: Non-Reproducible Items

The following main-paper figures and tables **cannot be replicated** with the available data:

### Requires Confidential CMA Data (hand-collected from 15 consultancies under NDA)
- **Table 2** — Consultant Capital Market Assumption summary statistics
- **Table 3** — Consultant Beliefs → Pension Allocations (OLS & IV)
- **Table 4** — Consultant Fixed Effects / Variance Decomposition
- **Figure 4a** — Median Consultant Alpha Over Time
- **Figure 4b** — Binscatter: Consultant Alpha vs Omega
- **Figure 5** — Distribution of Consultant Fixed Effects

### Requires IAR Data (Egan, Matvos & Seru 2019, updated — request to Mark Egan)
- **Table 3, cols 3–4** — Bartik IV specifications

### Requires PENDAT Surveys (1990s historical — PPCC/GFOA)
- **Table 5** — Experience Effects (state-level, 1990s equity entry timing)

### Requires Geocoding + CIO Tenure Data
- **Table 6** — Peer Effects (inverse-distance peer network + CIO tenure interactions)

### Requires External Institutional Data
- **Figure 7a/b** — Cross-institution comparison (NACUBO endowments, Milliman Corporate 100, UK PPF)

### Requires State Street GMP Data
- **Figure 8a/b** — Global Market Portfolio comparison + Active Share metric

### Requires CDC Mortality + BEA Funding Data
- **Table 8** — Covid Pandemic Spending Shock (also needs PPD Dec 2024 release)
- **Table 7, col 2** — BEA state-level funding ratio

### Requires Census QSPP Data
- **Figure 1b** — Historical risky share (1968–2021, PPD + QSPP overlay)

### Deferred (self-contained, can add later)
- **Table 9** — Simulation exercise (mean-variance model with 100K draws; no external data needed)